# 02 — COSMOS Feature Selection

Three-stage feature selection on the normalized expression matrices from notebook 01:

1. **Stage 1**: K-means modular reduction with Hopkins clustering tendency check
2. **Stage 2**: Dual outlier detection by consensus of IQR filtering and Isolation Forest
3. **Stage 3**: Modified Survival Laplacian Score for survival-relevant ranking

Stage 3 outputs feed the gene-reduction summary figure. The temporal transition
analysis (notebook 03) is run on ssGSEA scores of a pre-specified panel of 37
pathways defined here, independently of Stage 3.


In [ ]:
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler, quantile_transform

warnings.filterwarnings("ignore")
np.random.seed(42)

import os
BASE_DIR = Path(os.environ.get("COSMOS_BASE_DIR", "./data"))
INTER_DIR = BASE_DIR / "intermediates"


## Load preprocessed data


In [ ]:
def load_pkl(name):
    with open(INTER_DIR / f"{name}.pkl", "rb") as f:
        return pickle.load(f)


def save_pkl(obj, name):
    with open(INTER_DIR / f"{name}.pkl", "wb") as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)


geo_expr = load_pkl("geo_expr_corrected")
tcga_expr = load_pkl("tcga_expr_corrected")
geo_clinical = load_pkl("geo_clinical")
tcga_clinical = load_pkl("tcga_clinical")
ssgsea_corrected = load_pkl("ssgsea_corrected")
combined_genesets = load_pkl("combined_genesets")

# Align ssGSEA columns to clinical sample indices
all_clin = {**geo_clinical, **tcga_clinical}
for cohort, ss in list(ssgsea_corrected.items()):
    clin = all_clin.get(cohort)
    if ss is None or clin is None or clin.empty:
        continue
    common = [c for c in ss.columns.astype(str) if c in set(clin.index.astype(str))]
    if len(common) >= 20:
        ssgsea_corrected[cohort] = ss.loc[:, common]
        if cohort in geo_clinical:
            geo_clinical[cohort] = geo_clinical[cohort].loc[common]
        elif cohort in tcga_clinical:
            tcga_clinical[cohort] = tcga_clinical[cohort].loc[common]

all_expr = {**geo_expr, **tcga_expr}
print(f"Cohorts loaded: {sorted(all_expr.keys())}")


## Hopkins clustering tendency

Hopkins statistic on a feature space measures clustering tendency. Values close to
1 indicate well-separated clusters; values near 0.5 indicate uniform distribution.
We compute it across four feature subsets and a PCA projection, then report the
consensus median.


In [ ]:
def hopkins_core(X, n_samples, seed):
    """Single Hopkins statistic computation."""
    rng = np.random.RandomState(seed)
    n, d = X.shape
    ns = min(n_samples, n - 1)
    idx = rng.choice(n, ns, replace=False)
    X_s = X[idx]
    X_r = rng.uniform(X.min(0), X.max(0), (ns, d))
    nn = NearestNeighbors(n_neighbors=2).fit(X)
    u = nn.kneighbors(X_r, n_neighbors=1)[0].sum()
    w = nn.kneighbors(X_s, n_neighbors=2)[0][:, 1].sum()
    denom = u + w
    return float(u / denom) if denom > 1e-10 else 0.5


def hopkins_multimethod(X, n_runs=10):
    """Compute Hopkins across multiple feature subsets and a PCA projection.
    Returns a dict keyed by method with median, std, and the consensus median."""
    n_samples = min(max(int(X.shape[0] * 0.10), 50), 500)
    results = {}
    variances = X.var(axis=0)

    subsets = [
        ("M1_full", "Full feature space", X),
        ("M2_var50", "Top 50% variance",
         X[:, variances >= np.percentile(variances, 50)]),
        ("M3_var25", "Top 25% variance",
         X[:, variances >= np.percentile(variances, 75)]),
        ("M5_expressed", "Expressed genes",
         X[:, X.mean(axis=0) > 0] if (X.mean(axis=0) > 0).sum() > 50 else X),
    ]
    for key, label, X_sub in subsets:
        vals = [hopkins_core(X_sub, n_samples, s) for s in range(n_runs)]
        results[key] = {
            "label": label,
            "median": float(np.median(vals)),
            "std": float(np.std(vals)),
            "vals": vals,
        }

    n_pc = min(50, X.shape[0] - 1, X.shape[1] - 1)
    X_pca = PCA(n_components=n_pc, random_state=42).fit_transform(X)
    vals_pca = [hopkins_core(X_pca, n_samples, s) for s in range(n_runs)]
    results["M4_pca50"] = {
        "label": f"PCA top-{n_pc} components",
        "median": float(np.median(vals_pca)),
        "std": float(np.std(vals_pca)),
        "vals": vals_pca,
    }

    medians = [v["median"] for v in results.values()]
    results["_consensus"] = {
        "median": float(np.median(medians)),
        "std": float(np.std(medians)),
        "range": (float(np.min(medians)), float(np.max(medians))),
    }
    return results


## Stage 1: K-means modular reduction

For each cohort, cluster genes into modules and keep representative genes from each
cluster. The number of clusters is chosen by the elbow method within a cohort-specific
range. The retention rule keeps the top 20% closest to each cluster centroid.


In [ ]:
def find_optimal_k(X, k_min, k_max, n_probe=20):
    """Find optimal K by elbow method, probing n_probe values in [k_min, k_max]."""
    probe_ks = sorted(set(np.linspace(k_min, k_max, n_probe).astype(int)))
    sse = [
        KMeans(n_clusters=k, n_init=5, random_state=42,
               max_iter=200, algorithm="lloyd").fit(X).inertia_
        for k in probe_ks
    ]
    sse = np.array(sse)
    d2 = np.diff(np.diff(sse))
    opt_i = int(np.argmax(np.abs(d2))) + 1
    return int(np.clip(probe_ks[min(opt_i, len(probe_ks) - 1)], k_min, k_max)), sse, probe_ks


def stage1_kmeans(expr_df, name, k_min=150, k_max=250):
    """K-means modular reduction with Hopkins clustering check."""
    n_genes = expr_df.shape[0]
    X_gene = StandardScaler().fit_transform(expr_df.values)

    hop_pre = hopkins_multimethod(X_gene, n_runs=10)
    H_pre = hop_pre["_consensus"]["median"]

    opt_k, sse, probe_ks = find_optimal_k(X_gene, k_min, k_max)
    km = KMeans(
        n_clusters=opt_k, n_init=20, random_state=42,
        max_iter=500, algorithm="lloyd",
    )
    labels = km.fit_predict(X_gene)
    genes = expr_df.index.tolist()
    core_genes = []
    for k_idx in range(opt_k):
        mask = labels == k_idx
        cg = [genes[i] for i in range(len(genes)) if mask[i]]
        cx = X_gene[mask]
        dists = np.linalg.norm(cx - km.cluster_centers_[k_idx], axis=1)
        n_keep = max(1, int(np.floor(len(cg) * 0.20)))
        for si in np.argsort(dists)[:n_keep]:
            core_genes.append(cg[si])

    seen = set()
    core_genes = [g for g in core_genes if not (g in seen or seen.add(g))]

    core_expr = expr_df.loc[core_genes]
    X_core = StandardScaler().fit_transform(core_expr.values)
    H_post = hopkins_multimethod(X_core, n_runs=10)["_consensus"]["median"]

    print(f"  {name}: {len(genes)} -> {len(core_genes)} genes "
          f"(k={opt_k}, Hopkins {H_pre:.3f} -> {H_post:.3f})")
    return {
        "reduced_expr": core_expr,
        "core_genes": core_genes,
        "hopkins_pre": H_pre,
        "hopkins_post": H_post,
        "optimal_k": opt_k,
        "n_initial": len(genes),
        "n_reduced": len(core_expr),
    }


In [ ]:
print("Stage 1: K-means modular reduction")
stage1_results = {}
for name, expr in all_expr.items():
    arr = quantile_transform(
        expr.values, axis=0, output_distribution="normal", random_state=42
    )
    expr_norm = pd.DataFrame(arr, index=expr.index, columns=expr.columns)
    stage1_results[name] = stage1_kmeans(expr_norm, name)


## Stage 2: Dual outlier detection

Remove genes flagged by both IQR-based filtering and Isolation Forest. The intersection
rule preserves cancer-relevant features with extreme expression patterns (oncogene
amplifications, tumor suppressor silencing) that one method alone might discard.


In [ ]:
def stage2_outlier(s1_result, name, iqr_flag_frac=0.15, if_contamination=0.15):
    """Remove genes flagged by both IQR-based filtering and Isolation Forest."""
    expr = s1_result["reduced_expr"]
    vals = expr.values
    Q1 = np.percentile(vals, 25, axis=1)
    Q3 = np.percentile(vals, 75, axis=1)
    IQR_v = Q3 - Q1
    outside_frac = np.mean(
        (vals < (Q1 - 1.5 * IQR_v)[:, None])
        | (vals > (Q3 + 1.5 * IQR_v)[:, None]),
        axis=1,
    )
    iqr_cutoff = np.percentile(outside_frac, (1 - iqr_flag_frac) * 100)
    iqr_out = set(expr.index[outside_frac >= iqr_cutoff])

    X_genes = StandardScaler().fit_transform(expr.values)
    iso = IsolationForest(
        contamination=if_contamination,
        n_estimators=300, random_state=42, n_jobs=-1,
    )
    iso_out = set(expr.index[iso.fit_predict(X_genes) == -1])

    agreed = iqr_out & iso_out
    retained = [g for g in expr.index if g not in agreed]
    print(f"  {name}: removed {len(agreed)} genes, retained {len(retained)}")
    return {
        "retained_expr": expr.loc[retained],
        "n_input": expr.shape[0],
        "n_retained": len(retained),
        "n_removed": len(agreed),
    }


print("Stage 2: Dual outlier detection")
stage2_results = {
    name: stage2_outlier(s1, name) for name, s1 in stage1_results.items()
}


## Stage 3: Modified Survival Laplacian Score

Score each gene by how well it preserves the local manifold structure of survival
times. A Gaussian kernel weighted by censoring status assigns highest confidence to
pairs with both events observed, intermediate weight to informative mixed pairs
(one event, one censored at a longer time), and lowest weight to ambiguous censored
pairs.


In [ ]:
def survival_weight_matrix(time, event, alpha=4.82, beta=2.31, gamma=0.76):
    """Build a survival-aware affinity matrix. Weights follow the order
    alpha > beta > gamma to give highest confidence to fully-observed pairs."""
    t = np.asarray(time, float)
    e = np.asarray(event, float)
    td = np.abs(t[:, None] - t[None, :])
    sigma = np.median(td[td > 0]) + 1e-8
    K = np.exp(-td ** 2 / (2 * sigma ** 2))

    both_unc = (e[:, None] == 1) & (e[None, :] == 1)
    mixed = (
        ((e[:, None] == 1) & (e[None, :] == 0) & (t[None, :] > t[:, None]))
        | ((e[:, None] == 0) & (e[None, :] == 1) & (t[:, None] > t[None, :]))
    )
    both_cen = (e[:, None] == 0) & (e[None, :] == 0)

    W = alpha * K * both_unc + beta * K * mixed + gamma * K * both_cen
    np.fill_diagonal(W, 0)
    return W


def laplacian_score_batch(X, W, batch_size=500):
    """Batched Laplacian score. Lower scores indicate features that preserve
    the local manifold structure of the affinity graph."""
    n = X.shape[0]
    D = np.diag(W.sum(1))
    L = D - W
    D_sum = D.diagonal().sum()
    Dv = D.diagonal()
    scores = np.full(n, np.inf)
    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        F = X[start:end]
        f_mean = (F * Dv[None, :]).sum(1) / D_sum
        Fc = F - f_mean[:, None]
        num = np.einsum("ij,jk,ik->i", Fc, L, Fc)
        den = np.einsum("ij,j,ij->i", Fc, Dv, Fc)
        with np.errstate(divide="ignore", invalid="ignore"):
            scores[start:end] = np.where(den > 1e-10, num / den, np.inf)
    return scores


def stage3_mslr(s2_result, clinical, name, top_pct=0.20):
    """Rank genes by Modified Survival Laplacian Score and keep the top
    fraction per cohort."""
    expr = s2_result["retained_expr"]
    common = sorted(set(expr.columns) & set(clinical.index))
    if len(common) < 20:
        return None
    expr_a = expr[common]
    clin_a = clinical.loc[common]
    n_evt = int(clin_a["event"].sum())
    if n_evt < 10:
        print(f"  {name}: only {n_evt} events, MSLR unstable")
        return None
    W = survival_weight_matrix(clin_a["time"].values, clin_a["event"].values)
    X = StandardScaler().fit_transform(expr_a.values.T).T
    scores = laplacian_score_batch(X, W)
    gene_scores = (
        pd.Series(scores, index=expr_a.index)
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        .sort_values()
    )
    n_sel = max(int(len(gene_scores) * top_pct), 20)
    selected = gene_scores.index[:n_sel].tolist()
    print(f"  {name}: selected {n_sel} genes (top {top_pct*100:.0f}%)")
    return {
        "selected_genes": selected,
        "selected_expr": expr_a.loc[selected],
        "gene_scores": gene_scores,
        "n_input": expr_a.shape[0],
        "n_selected": n_sel,
    }


print("Stage 3: MSLR ranking")
stage3_results = {}
for name in stage2_results:
    clin = geo_clinical.get(name, tcga_clinical.get(name))
    if clin is not None and not clin.empty:
        r = stage3_mslr(stage2_results[name], clin, name)
        if r is not None:
            stage3_results[name] = r


## Pre-specified 37-pathway panel

The transition discovery is run on ssGSEA enrichment scores for a panel of 37
pathways curated *a priori* from the Hallmark and KEGG legacy collections to span
four biological modules. This panel was fixed before any survival analysis to
prevent post-hoc selection bias.


In [ ]:
PATHWAY_MODULES_37 = {
    "Proliferation": [
        "HALLMARK_E2F_TARGETS",
        "HALLMARK_G2M_CHECKPOINT",
        "HALLMARK_MYC_TARGETS_V1",
        "HALLMARK_MYC_TARGETS_V2",
        "HALLMARK_MITOTIC_SPINDLE",
        "HALLMARK_DNA_REPAIR",
        "KEGG_CELL_CYCLE",
        "KEGG_DNA_REPLICATION",
        "KEGG_MISMATCH_REPAIR",
        "KEGG_P53_SIGNALING_PATHWAY",
        "KEGG_HOMOLOGOUS_RECOMBINATION",
    ],
    "Immune": [
        "HALLMARK_INTERFERON_ALPHA_RESPONSE",
        "HALLMARK_INTERFERON_GAMMA_RESPONSE",
        "HALLMARK_INFLAMMATORY_RESPONSE",
        "HALLMARK_IL6_JAK_STAT3_SIGNALING",
        "HALLMARK_TNFA_SIGNALING_VIA_NFKB",
        "HALLMARK_COMPLEMENT",
        "KEGG_NATURAL_KILLER_CELL_MEDIATED_CYTOTOXICITY",
        "KEGG_T_CELL_RECEPTOR_SIGNALING_PATHWAY",
        "KEGG_ANTIGEN_PROCESSING_AND_PRESENTATION",
    ],
    "Metabolic": [
        "HALLMARK_OXIDATIVE_PHOSPHORYLATION",
        "HALLMARK_GLYCOLYSIS",
        "HALLMARK_ADIPOGENESIS",
        "HALLMARK_BILE_ACID_METABOLISM",
        "HALLMARK_CHOLESTEROL_HOMEOSTASIS",
        "HALLMARK_XENOBIOTIC_METABOLISM",
        "KEGG_CITRATE_CYCLE_TCA_CYCLE",
        "KEGG_GLYCOLYSIS_GLUCONEOGENESIS",
    ],
    "Microenvironment": [
        "HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION",
        "HALLMARK_ANGIOGENESIS",
        "HALLMARK_HYPOXIA",
        "HALLMARK_APOPTOSIS",
        "HALLMARK_COAGULATION",
        "HALLMARK_TGF_BETA_SIGNALING",
        "HALLMARK_WNT_BETA_CATENIN_SIGNALING",
        "HALLMARK_NOTCH_SIGNALING",
        "KEGG_ECM_RECEPTOR_INTERACTION",
    ],
}

ALL_37 = [p for ps in PATHWAY_MODULES_37.values() for p in ps]
PW_TO_MOD = {p: m for m, ps in PATHWAY_MODULES_37.items() for p in ps}
print(f"Panel size: {len(ALL_37)} pathways across {len(PATHWAY_MODULES_37)} modules")


## Module scores from ssGSEA

For each cohort, average the ssGSEA NES across pathways within each module to obtain
per-sample module scores.


In [ ]:
def module_scores_from_ssgsea(ss, modules):
    """Compute mean ssGSEA NES per module per sample."""
    if ss is None:
        return None
    ms = {}
    for mod, pws in modules.items():
        avail = [p for p in pws if p in ss.index]
        if avail:
            ms[mod] = ss.loc[avail].apply(pd.to_numeric, errors="coerce").mean(0)
    return pd.DataFrame(ms).T if ms else None


module_scores_37 = {
    name: module_scores_from_ssgsea(ss, PATHWAY_MODULES_37)
    for name, ss in ssgsea_corrected.items()
}

for name, ms in module_scores_37.items():
    if ms is not None:
        print(f"  {name}: {ms.shape[0]} modules x {ms.shape[1]} samples")


## Cancer-specific module resolution

Resolve cancer-type-specific pathway names (hormone signaling, RAS/MAPK, NRF2,
etc.) against the actual MSigDB collection. Handles MSigDB version drift where
pathway names may differ slightly between releases.


In [ ]:
ALL_GS_KEYS = set(combined_genesets.keys())

STOP_WORDS = frozenset({
    "THE", "AND", "OF", "BY", "IN", "VIA", "TO", "AT",
    "AN", "A", "FOR", "WITH", "FROM",
})


def name_tokens(name):
    """Upper-case tokens with stop-words and short tokens removed."""
    return [
        t for t in name.upper().replace("-", "_").split("_")
        if len(t) > 2 and t not in STOP_WORDS
    ]


MANUAL_OVERRIDES = {
    "REACTOME_VEGFA_VEGFR2_SIGNALING": [
        "REACTOME_SIGNALING_BY_VEGF",
        "GOBP_VASCULAR_ENDOTHELIAL_GROWTH_FACTOR_RECEPTOR_SIGNALING_PATHWAY",
    ],
    "REACTOME_NRF2_TARGETS": [
        "REACTOME_NFE2L2_REGULATING_ANTIOXIDANT_GENES",
        "REACTOME_KEAP1_NFE2L2_PATHWAY",
        "GOBP_RESPONSE_TO_OXIDATIVE_STRESS",
    ],
    "REACTOME_SIGNALING_BY_PI3K_AKT": [
        "REACTOME_PI3K_CASCADE",
        "REACTOME_PI3K_EVENTS_IN_ERBB2_SIGNALING",
        "GOBP_PHOSPHATIDYLINOSITOL_3_KINASE_SIGNALING",
    ],
}


def resolve_pathway_name(intended, all_keys, min_score=2, n_candidates=5):
    """Resolve a pathway name against an MSigDB collection.
    Returns (resolved_name, is_exact, top_candidates)."""
    if intended in all_keys:
        return intended, True, [(intended, len(name_tokens(intended)))]

    if intended in MANUAL_OVERRIDES:
        for fallback in MANUAL_OVERRIDES[intended]:
            if fallback in all_keys:
                return fallback, False, [(fallback, 99)]

    toks = name_tokens(intended)
    if not toks:
        return None, False, []

    scored = {}
    for key in all_keys:
        sc = sum(1 for t in toks if t in key.upper())
        if sc > 0:
            scored[key] = sc
    top = sorted(scored.items(), key=lambda x: -x[1])[:n_candidates]
    if not top or top[0][1] < min_score:
        return None, False, top
    return top[0][0], False, top


In [ ]:
BRCA_INTENDED = {
    "Hormone_signalling": [
        "HALLMARK_ESTROGEN_RESPONSE_EARLY",
        "HALLMARK_ESTROGEN_RESPONSE_LATE",
        "HALLMARK_ANDROGEN_RESPONSE",
        "REACTOME_SIGNALING_BY_ERBB2",
    ],
    "PI3K_AKT_resistance": [
        "REACTOME_PI3K_AKT_SIGNALING_IN_CANCER",
        "REACTOME_SIGNALING_BY_PI3K_AKT",
        "REACTOME_FOXO_MEDIATED_TRANSCRIPTION",
        "HALLMARK_REACTIVE_OXYGEN_SPECIES_PATHWAY",
    ],
    "Oncogenic_RTK": [
        "REACTOME_SIGNALING_BY_EGFR",
        "REACTOME_DOWNSTREAM_SIGNAL_TRANSDUCTION",
        "KEGG_ERBB_SIGNALING_PATHWAY",
        "REACTOME_VEGFA_VEGFR2_SIGNALING",
    ],
}

LUAD_INTENDED = {
    "RAS_MAPK": [
        "HALLMARK_KRAS_SIGNALING_UP",
        "HALLMARK_KRAS_SIGNALING_DN",
        "REACTOME_RAF_MAP_KINASE_CASCADE",
        "REACTOME_MAP2K_AND_MAPK_ACTIVATION",
    ],
    "NRF2_oxidative_stress": [
        "HALLMARK_REACTIVE_OXYGEN_SPECIES_PATHWAY",
        "KEGG_GLUTATHIONE_METABOLISM",
        "REACTOME_NRF2_TARGETS",
        "HALLMARK_FATTY_ACID_METABOLISM",
    ],
    "RTK_driver_signalling": [
        "REACTOME_SIGNALING_BY_EGFR",
        "REACTOME_SIGNALING_BY_EGFR_IN_CANCER",
        "HALLMARK_HEDGEHOG_SIGNALING",
        "REACTOME_SIGNALING_BY_ALK",
    ],
}


def resolve_modules(intended_dict, all_keys, label, core_set):
    """Resolve all pathways in intended_dict, deduplicating within each module
    and excluding pathways already in the core 37 panel."""
    resolved = {}
    n_exact, n_fuzzy, n_dropped = 0, 0, 0
    print(f"  {label}:")
    for mod, pathways in intended_dict.items():
        mod_paths, seen_in_mod = [], set()
        for pw in pathways:
            if pw in core_set:
                n_dropped += 1
                continue
            res, is_exact, _ = resolve_pathway_name(pw, all_keys)
            if res is None or res in seen_in_mod or res in core_set:
                n_dropped += 1
                continue
            mod_paths.append(res)
            seen_in_mod.add(res)
            if is_exact:
                n_exact += 1
            else:
                n_fuzzy += 1
        if mod_paths:
            resolved[mod] = mod_paths
            print(f"    {mod}: {len(mod_paths)} pathways")
    print(f"  ({n_exact} exact, {n_fuzzy} resolved, {n_dropped} dropped)")
    return resolved


CORE_37_SET = set(ALL_37)
BRCA_RESOLVED = resolve_modules(BRCA_INTENDED, ALL_GS_KEYS, "BRCA-specific", CORE_37_SET)
LUAD_RESOLVED = resolve_modules(LUAD_INTENDED, ALL_GS_KEYS, "LUAD-specific", CORE_37_SET)


In [ ]:
def compute_cancer_module_scores(ss_dict, module_dict):
    """Compute mean NES per cancer-specific module per cohort."""
    out = {}
    for cohort, ss in ss_dict.items():
        if ss is None:
            continue
        scores = module_scores_from_ssgsea(ss, module_dict)
        if scores is not None and not scores.empty:
            out[cohort] = scores
    return out


module_scores_brca_specific = compute_cancer_module_scores(ssgsea_corrected, BRCA_RESOLVED)
module_scores_luad_specific = compute_cancer_module_scores(ssgsea_corrected, LUAD_RESOLVED)
print(f"BRCA-specific scores: {sorted(module_scores_brca_specific.keys())}")
print(f"LUAD-specific scores: {sorted(module_scores_luad_specific.keys())}")


## Save outputs


In [ ]:
save_pkl(stage1_results, "stage1_results")
save_pkl(stage2_results, "stage2_results")
save_pkl(stage3_results, "stage3_results")
save_pkl(PATHWAY_MODULES_37, "core_pathways_37")
save_pkl(module_scores_37, "module_scores_37")
save_pkl(BRCA_RESOLVED, "brca_specific_modules")
save_pkl(LUAD_RESOLVED, "luad_specific_modules")
save_pkl(module_scores_brca_specific, "module_scores_brca_specific")
save_pkl(module_scores_luad_specific, "module_scores_luad_specific")

print("Next: 03_msrs_phase_analysis.ipynb")
